<a href="https://colab.research.google.com/github/MatchLab-Imperial/deep-learning-course/blob/master/10_Data_Classification_YOLO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏠 Data collection and classification Challenge — YOLOv8

**Objective:** You have been hired as an AI expert and your new boss, who knows nothing about AI, asked you to quickly develop a DL model that can classify roofs in the UK from aerial views. The input is in the form of image crops from Google satellite views, each containing one building. Some crops may contain a few buildings but it can be considered as noise. You should collect data and finetune `yolov8n-cls.pt`  model.   Classify aerial roof images into **flat** or **pitched**. These are the constraints, otherwise you are free to make your own design choices.

- Tune training hyperparameters  
- Add more images to improve your model  
- Experiment with augmentation or optimization settings  

When you are satisfied, submit the annotated data `*CID*_data.zip` with the same file structure as the provided sample as well as your best `*CID*_best.pt` checkpoint, which will be evaluated on a hidden test set. Keep the filenames of the existing samples as they are.

---

### 📁 Expected dataset structure
<pre>
.../aerial
├─ train/
│ ├─ flat/
│ └─ pitched/
├─ val/
│ ├─ flat/
│ └─ pitched/
└─ test/
  ├─ flat/
  └─ pitched/
</pre>

---

### 📌 Notes

- You will train only the classification model `yolov8n-cls.pt`.

- Image size (`imgsz`):  
  - YOLOv8 automatically resizes any input to the `imgsz` you set in `model.train(...)`.  
  - Tip: using sizes that are multiples of 32 (224, 256, 320, 384, …) aligns well with the backbone. Larger sizes train slower, but can perform better.
  - When collecting images, there’s no strict size rule; just aim for **clear roofs**.

- Collecting more images:
  - Sources: **Google Earth**, **OpenAerialMap**, **Mapillary**, **Kaggle**, public GIS/satellite portals, or your **own drone/photos**.  
  - Prefer images that aren’t tiny/blurry/over-compressed. RGB `.jpg/.jpeg/.png/.bmp/.webp` are all fine.

- Adding new data to `/aerial`:
  1. Label by roof type and drop files into the correct folders locally:
     ```
     aerial/
     ├─ train/
     │  ├─ flat/
     │  └─ pitched/
     ├─ val/
     │  ├─ flat/
     │  └─ pitched/
     └─ test/
        ├─ flat/
        └─ pitched/
     ```
  2. Do **not** duplicate the same image across `train/`, `val/`, and `test/`.
  3. Compress local folder to [aerial.zip](https://github.com/MatchLab-Imperial/deep-learning-course/blob/master/asset/10_Classification_YOLO/aerial.zip) to upload to colab in the code block below.



In [ ]:
# !pip install -q ultralytics  # uncomment if not installed

from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random, torch

from google.colab import files
uploaded = files.upload()  # choose aerial.zip

In [ ]:
!unzip -q -o aerial.zip -d /content/
!find /content/aerial -maxdepth 2 -type d -print  # quick check

In [ ]:
DATA_ROOT = Path("/content/aerial")
assert (DATA_ROOT/"train").exists() and (DATA_ROOT/"val").exists(), "train/val folders missing"

# List classes from train/
classes = sorted([p.name for p in (DATA_ROOT/"train").iterdir() if p.is_dir()])
print("Classes:", classes)

# Collect sample images
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
samples = []
for cls in classes:
    samples += [p for p in (DATA_ROOT/"train"/cls).iterdir() if p.suffix.lower() in exts]

random.shuffle(samples)
show = samples[:6]

# Plot
cols = 3
rows = 2
plt.figure(figsize=(12, 6))
for i, p in enumerate(show, 1):
    img = Image.open(p)
    plt.subplot(rows, cols, i)
    plt.imshow(img)
    plt.title(p.parent.name)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 🛠 Training Tips — Things You Can Try

You may experiment with these arguments in `model.train()`:

| Parameter | Meaning | Suggested values |
|-----------|---------|------------------|
| `imgsz` | input image size | `224`, `320`, `384` |
| `epochs` | training duration | `10–60` |
| `batch` | batch size | `16–128` |
| `auto_augment` | built-in augmentation | `"randaugment"`, `"ta_wide"` |
| `mixup` / `cutmix` | label mixing | `0.0–0.3` |
| `erasing` | random erasing | `0.0–0.7` |
| `optimizer` | optimizer choice | `"SGD"` or `"AdamW"` |
| `cos_lr` | cosine LR schedule | `True` or `False` |

Tip: only change one or two things at a time, and always monitor `top1_acc`. (`top5_acc` is printed automatically in YOLO classification tasks, but is useless in our two-class task.)


In [ ]:
DATA_ROOT = Path("/content/aerial")
assert (DATA_ROOT/"train").exists(), "train folder missing"
assert (DATA_ROOT/"val").exists(), "val folder missing"
assert (DATA_ROOT/"test").exists(), "test folder missing"

device = 0 if torch.cuda.is_available() else "cpu"

# Loading base model
model = YOLO("yolov8n-cls.pt")

# ======== TRAIN (you may edit arguments below) ========
results = model.train(
    data=str(DATA_ROOT),     # uses train/ and val/
    imgsz=320,               # try 224/320/384
    epochs=30,               # increase if val improves
    batch=64,
    device=device,
    workers=2,
    patience=10,
    auto_augment="randaugment",
    erasing=0.5,
    mixup=0.1,
    cutmix=0.1,
    # optimizer="AdamW",
    # cos_lr=True,
    verbose=False,           # <-- minimising console output
    plots=False,             # <-- skipping plots
)

best_ckpt = model.trainer.best
print("Best checkpoint to submit:", best_ckpt)

# ======== TEST (held-out split) ========
metrics = model.val(
    data=str(DATA_ROOT),
    split="test",
    imgsz=320,
    device=device,
    verbose=False,           # <-- minimising console output
)
print(f"Test: {100*metrics.top1:.4f}%")


### 💾 Saving and Submitting Your Model

Once training is complete, you’ll download your best-performing model checkpoint for submission.  
Follow the short prompt below to enter your **CID** — this will automatically rename your `best.pt` file (to `*CID*_best.pt`) and trigger the download.

In [ ]:
import re, shutil

# Ensuring best_ckpt exists
assert 'best_ckpt' in globals(), "Run the training cell first to define `best_ckpt`."
src = Path(best_ckpt)
assert src.exists(), f"Checkpoint not found at: {src}"


cid = input("Enter your College ID: ").strip()
cid = re.sub(r'[^0-9]', '', cid)
if not cid:
    raise ValueError("Invalid College ID. Please use digits only (0–9).")


dst = Path(f"/content/{cid}_best.pt")
shutil.copy(src, dst)
print(f"Prepared submission file: {dst}")

# Downloading
files.download(str(dst))

In [ ]:
###############################################################################
# CELL 0 — README (Markdown cell in Colab)
###############################################################################
# # 🏠 Roof Classification with YOLOv8 — Complete Pipeline
#
# This notebook fine-tunes `yolov8n-cls.pt` to classify aerial roof images
# as **flat** or **pitched**.
#
# **Instructions:**
# 1. Set runtime to **GPU** (Runtime → Change runtime → T4 GPU)
# 2. Prepare your `aerial.zip` with the correct folder structure
# 3. Run all cells in order
# 4. Download your `CID_best.pt` and `CID_data.zip` at the end

In [ ]:
###############################################################################
# CELL 1 — Install dependencies
###############################################################################
!pip install -q ultralytics

In [ ]:
###############################################################################
# CELL 2 — Imports
###############################################################################
from ultralytics import YOLO
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import random, torch, shutil, re, os
 
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:             {torch.cuda.get_device_name(0)}")

In [ ]:
###############################################################################
# CELL 3 — Upload your aerial.zip
###############################################################################
from google.colab import files
 
print("📤 Please upload your aerial.zip file...")
uploaded = files.upload()  # Select aerial.zip from your local machine

In [ ]:
###############################################################################
# CELL 4 — Extract dataset
###############################################################################
import zipfile
 
zip_path = "/content/aerial.zip"
extract_to = "/content/"
 
# Handle case where upload puts file in current directory
if not os.path.exists(zip_path):
    for name in uploaded.keys():
        if name.endswith(".zip"):
            shutil.move(name, zip_path)
            break
 
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_to)
 
print("✅ Extracted successfully.")

In [ ]:
###############################################################################
# CELL 5 — Verify dataset structure
###############################################################################
DATA_ROOT = Path("/content/aerial")
 
# Check required folders exist
required = ["train/flat", "train/pitched", "val/flat", "val/pitched",
            "test/flat", "test/pitched"]
for folder in required:
    path = DATA_ROOT / folder
    assert path.exists(), f"❌ Missing folder: {path}"
print("✅ All required folders found.\n")
 
# Count images per split/class
exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
 
print("📊 Dataset summary:")
print("-" * 40)
total = 0
for split in ["train", "val", "test"]:
    for cls in ["flat", "pitched"]:
        folder = DATA_ROOT / split / cls
        count = len([f for f in folder.iterdir() if f.suffix.lower() in exts])
        total += count
        print(f"  {split:5s}/{cls:8s}: {count:4d} images")
print("-" * 40)
print(f"  {'TOTAL':14s}: {total:4d} images\n")
 
# List classes
classes = sorted([p.name for p in (DATA_ROOT / "train").iterdir() if p.is_dir()])
print(f"Classes: {classes}")

In [ ]:
###############################################################################
# CELL 6 — Visualize sample images
###############################################################################
samples = []
for cls in classes:
    samples += [p for p in (DATA_ROOT / "train" / cls).iterdir()
                if p.suffix.lower() in exts]
 
random.shuffle(samples)
show = samples[:min(6, len(samples))]
 
cols = 3
rows = (len(show) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12, 4 * rows))
axes = axes.flatten() if rows > 1 else [axes] if rows == 1 and cols == 1 else axes.flatten()
 
for i, ax in enumerate(axes):
    if i < len(show):
        img = Image.open(show[i])
        ax.imshow(img)
        ax.set_title(show[i].parent.name, fontsize=12, fontweight="bold")
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
###############################################################################
# CELL 7 — Training configuration
###############################################################################
# ╔══════════════════════════════════════════════════════════════════╗
# ║  EDIT THESE PARAMETERS TO EXPERIMENT                           ║
# ╚══════════════════════════════════════════════════════════════════╝
 
CONFIG = {
    "imgsz":        224,              # Image size: 224 / 320 / 384
    "epochs":       50,               # Max training epochs
    "batch":        64,               # Batch size (reduce to 32 if OOM)
    "patience":     10,               # Early stopping patience
    "optimizer":    "AdamW",          # "SGD" or "AdamW"
    "cos_lr":       True,             # Cosine learning rate schedule
    "lr0":          0.001,            # Initial learning rate
    "auto_augment": "randaugment",    # "randaugment" or "ta_wide"
    "erasing":      0.4,              # Random erasing probability
    "mixup":        0.1,              # Mixup alpha
    "cutmix":       0.1,              # CutMix alpha
}
 
print("🔧 Training configuration:")
for k, v in CONFIG.items():
    print(f"  {k:15s}: {v}")

In [ ]:
###############################################################################
# CELL 8 — Train the model
###############################################################################
device = 0 if torch.cuda.is_available() else "cpu"
print(f"\n🚀 Training on device: {device}")
 
# Load pretrained YOLOv8-nano classification model
model = YOLO("yolov8n-cls.pt")
 
# Train
results = model.train(
    data=str(DATA_ROOT),
    imgsz=CONFIG["imgsz"],
    epochs=CONFIG["epochs"],
    batch=CONFIG["batch"],
    patience=CONFIG["patience"],
    optimizer=CONFIG["optimizer"],
    cos_lr=CONFIG["cos_lr"],
    lr0=CONFIG["lr0"],
    auto_augment=CONFIG["auto_augment"],
    erasing=CONFIG["erasing"],
    mixup=CONFIG["mixup"],
    cutmix=CONFIG["cutmix"],
    device=device,
    workers=2,
    verbose=True,
    plots=True,
)
 
best_ckpt = model.trainer.best
print(f"\n✅ Training complete!")
print(f"📁 Best checkpoint: {best_ckpt}")

In [ ]:
###############################################################################
# CELL 9 — Evaluate on your test split
###############################################################################
print("\n📊 Evaluating on test split...")
metrics = model.val(
    data=str(DATA_ROOT),
    split="test",
    imgsz=CONFIG["imgsz"],
    device=device,
    verbose=True,
)
 
test_acc = 100 * metrics.top1
print(f"\n🎯 Test Top-1 Accuracy: {test_acc:.2f}%")
 
if test_acc < 80:
    print("⚠️  Accuracy is below 80%. Consider:")
    print("    - Adding more training images")
    print("    - Trying imgsz=320 or 384")
    print("    - Training for more epochs")
    print("    - Checking for mislabeled images")
elif test_acc < 90:
    print("👍 Decent accuracy. You can likely improve with more data or tuning.")
else:
    print("🏆 Excellent accuracy!")

In [ ]:
###############################################################################
# CELL — Hyperparameter Experiments (run AFTER your baseline training)
###############################################################################
from ultralytics import YOLO
from pathlib import Path
import torch

DATA_ROOT = Path("/content/aerial")
device = 0 if torch.cuda.is_available() else "cpu"

# ╔══════════════════════════════════════════════════════════════════╗
# ║  5 experiments, each varying 1-2 params from baseline          ║
# ║  Baseline: imgsz=224, AdamW, cos_lr, randaugment, erasing=0.4  ║
# ╚══════════════════════════════════════════════════════════════════╝

experiments = {
    "exp1_imgsz320": {
        "desc": "Higher resolution (320 vs 224)",
        "imgsz": 320, "batch": 32,  # smaller batch to fit memory
        # rest = baseline defaults
        "optimizer": "AdamW", "cos_lr": True, "auto_augment": "randaugment",
        "erasing": 0.4, "mixup": 0.1, "cutmix": 0.1,
    },
    "exp2_sgd": {
        "desc": "SGD optimizer instead of AdamW",
        "imgsz": 224, "batch": 64,
        "optimizer": "SGD", "cos_lr": True, "auto_augment": "randaugment",
        "erasing": 0.4, "mixup": 0.1, "cutmix": 0.1,
    },
    "exp3_no_augment": {
        "desc": "No mixup/cutmix/erasing (ablation)",
        "imgsz": 224, "batch": 64,
        "optimizer": "AdamW", "cos_lr": True, "auto_augment": "randaugment",
        "erasing": 0.0, "mixup": 0.0, "cutmix": 0.0,
    },
    "exp4_heavy_augment": {
        "desc": "Heavy augmentation (erasing=0.7, mixup=0.3)",
        "imgsz": 224, "batch": 64,
        "optimizer": "AdamW", "cos_lr": True, "auto_augment": "augmix",
        "erasing": 0.7, "mixup": 0.3, "cutmix": 0.0,
    },
    "exp5_imgsz384_sgd": {
        "desc": "Largest resolution + SGD",
        "imgsz": 384, "batch": 16,
        "optimizer": "SGD", "cos_lr": True, "auto_augment": "randaugment",
        "erasing": 0.4, "mixup": 0.1, "cutmix": 0.1,
    },
}

# Store results
results_log = []

# Add your baseline result manually
results_log.append({
    "name": "baseline",
    "desc": "imgsz=224, AdamW, cos_lr, randaugment, erasing=0.4",
    "val_acc": 100.0,
    "test_acc": 98.44,
    "epochs_run": 20,
})

for name, cfg in experiments.items():
    print(f"\n{'='*60}")
    print(f"🧪 {name}: {cfg['desc']}")
    print(f"{'='*60}")

    model = YOLO("yolov8n-cls.pt")

    model.train(
        data=str(DATA_ROOT),
        imgsz=cfg["imgsz"],
        epochs=50,
        batch=cfg["batch"],
        patience=10,
        optimizer=cfg["optimizer"],
        cos_lr=cfg["cos_lr"],
        lr0=0.001,
        auto_augment=cfg["auto_augment"],
        erasing=cfg["erasing"],
        mixup=cfg["mixup"],
        cutmix=cfg["cutmix"],
        device=device,
        workers=2,
        verbose=False,
        plots=True,
        name=name,
    )

    # Get val accuracy from training
    val_acc = 100 * model.trainer.metrics.get("metrics/accuracy_top1", 0)

    # Evaluate on test split
    test_metrics = model.val(
        data=str(DATA_ROOT),
        split="test",
        imgsz=cfg["imgsz"],
        device=device,
        verbose=False,
    )
    test_acc = 100 * test_metrics.top1
    epochs_run = model.trainer.epoch + 1

    results_log.append({
        "name": name,
        "desc": cfg["desc"],
        "val_acc": val_acc,
        "test_acc": test_acc,
        "epochs_run": epochs_run,
    })

    print(f"  ✅ Val: {val_acc:.2f}% | Test: {test_acc:.2f}% | Epochs: {epochs_run}")

# ╔══════════════════════════════════════════════════════════════════╗
# ║  Summary table                                                  ║
# ╚══════════════════════════════════════════════════════════════════╝
print(f"\n{'='*75}")
print(f"📊 EXPERIMENT SUMMARY")
print(f"{'='*75}")
print(f"{'Experiment':<22} {'Description':<35} {'Val%':>6} {'Test%':>6} {'Ep':>4}")
print(f"{'-'*75}")
for r in results_log:
    print(f"{r['name']:<22} {r['desc'][:35]:<35} {r['val_acc']:>5.1f}% {r['test_acc']:>5.1f}% {r['epochs_run']:>4}")
print(f"{'='*75}")

# Find best
best = max(results_log, key=lambda x: x["test_acc"])
print(f"\n🏆 Best experiment: {best['name']} (Test: {best['test_acc']:.2f}%)")

In [ ]:
###############################################################################
# CELL — Visualize all experiment training curves + download
###############################################################################
import numpy as np
from pathlib import Path
import shutil

run_base = Path("/content/runs/classify")

exp_folders = {
    "baseline":           run_base / "train",
    "exp1_imgsz320":      run_base / "exp1_imgsz3203",
    "exp2_sgd":           run_base / "exp2_sgd3",
    "exp3_no_augment":    run_base / "exp3_no_augment3",
    "exp4_heavy_augment": run_base / "exp4_heavy_augment3",
    "exp5_imgsz384_sgd":  run_base / "exp5_imgsz384_sgd2",
}

for exp_name, exp_dir in exp_folders.items():
    if not exp_dir.exists():
        print(f"⚠️  Skipping {exp_name} — folder not found")
        continue

    plots = list(exp_dir.glob("*.png"))
    if not plots:
        print(f"⚠️  Skipping {exp_name} — no plots found")
        continue

    print(f"\n{'='*50}")
    print(f"📊 {exp_name}")
    print(f"{'='*50}")

    fig, axes = plt.subplots(
        1, min(3, len(plots)),
        figsize=(6 * min(3, len(plots)), 5)
    )
    if not isinstance(axes, (list, tuple, np.ndarray)):
        axes = [axes]
    else:
        axes = np.array(axes).flatten()

    for i, ax in enumerate(axes):
        if i < len(plots):
            img = Image.open(plots[i])
            ax.imshow(img)
            ax.set_title(f"{exp_name} — {plots[i].stem}", fontsize=9)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

    # Download with experiment name in filename
    for plot_path in plots:
        renamed = Path(f"/content/{exp_name}_{plot_path.name}")
        shutil.copy(plot_path, renamed)
        files.download(str(renamed))
        print(f"  📥 Downloaded: {renamed.name}")

In [ ]:
###############################################################################
# CELL 11 — Quick prediction sanity check
###############################################################################
# Run predictions on a few test images to visually confirm
test_imgs = []
for cls in classes:
    folder = DATA_ROOT / "test" / cls
    imgs = [f for f in folder.iterdir() if f.suffix.lower() in exts]
    test_imgs.extend(imgs[:3])
 
if test_imgs:
    print("🔍 Sample predictions on test images:")
    best_model = YOLO(str(best_ckpt))
    for img_path in test_imgs[:6]:
        result = best_model.predict(str(img_path), imgsz=CONFIG["imgsz"],
                                     verbose=False)
        pred_class = result[0].names[result[0].probs.top1]
        confidence = result[0].probs.top1conf.item()
        true_class = img_path.parent.name
        status = "✅" if pred_class == true_class else "❌"
        print(f"  {status} True: {true_class:8s} | Pred: {pred_class:8s} "
              f"| Conf: {confidence:.2%} | {img_path.name}")

In [ ]:
###############################################################################
# CELL 12 — Prepare submission: rename checkpoint
###############################################################################
assert 'best_ckpt' in globals(), "❌ Run the training cell first."
src = Path(best_ckpt)
assert src.exists(), f"❌ Checkpoint not found at: {src}"
 
cid = input("Enter your College ID (digits only): ").strip()
cid = re.sub(r'[^0-9]', '', cid)
if not cid:
    raise ValueError("❌ Invalid College ID. Use digits only.")
 
# Copy and rename best.pt
dst_model = Path(f"/content/{cid}_best.pt")
shutil.copy(src, dst_model)
print(f"✅ Model saved as: {dst_model}")

In [ ]:
###############################################################################
# CELL 13 — Prepare submission: zip dataset
###############################################################################
dst_data = Path(f"/content/{cid}_data")
 
# Create zip of the aerial folder
shutil.make_archive(str(dst_data), 'zip', '/content/', 'aerial')
print(f"✅ Dataset saved as: {dst_data}.zip")

In [ ]:
###############################################################################
# CELL 14 — Download submission files
###############################################################################
print("\n📥 Downloading submission files...")
files.download(str(dst_model))
files.download(f"{dst_data}.zip")
print("\n🎉 Done! Submit both files:")
print(f"  1. {dst_model.name}")
print(f"  2. {dst_data.name}.zip")

In [ ]:
✅ All required folders found.

📊 Dataset summary:
----------------------------------------
  train/flat    :  146 images
  train/pitched :  146 images
  val  /flat    :   32 images
  val  /pitched :   32 images
  test /flat    :   32 images
  test /pitched :   32 images
----------------------------------------
  TOTAL         :  420 images

Classes: ['flat', 'pitched']


In [ ]:
🔧 Training configuration:
  imgsz          : 224
  epochs         : 50
  batch          : 64
  patience       : 10
  optimizer      : AdamW
  cos_lr         : True
  lr0            : 0.001
  auto_augment   : randaugment
  erasing        : 0.4
  mixup          : 0.1
  cutmix         : 0.1


In [ ]:

🚀 Training on device: 0
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolov8n-cls.pt to 'yolov8n-cls.pt': 100% ━━━━━━━━━━━━ 5.3MB 226.3MB/s 0.0s
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.1, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/train, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=True, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
Downloading https://github.com/ultralytics/assets/releases/download/v8.4.0/yolo26n.pt to 'yolo26n.pt': 100% ━━━━━━━━━━━━ 5.3MB 336.0MB/s 0.0s
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2670.9±1925.8 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 1.9Kit/s 0.2s
train: New cache created: /content/aerial/train.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2246.5±1907.3 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 1.9Kit/s 0.0s
val: New cache created: /content/aerial/val.cache
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/train
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
Downloading https://ultralytics.com/assets/Arial.ttf to '/root/.config/Ultralytics/Arial.ttf': 100% ━━━━━━━━━━━━ 755.1KB 127.3MB/s 0.0s
       1/50     0.816G      0.652         36        224: 100% ━━━━━━━━━━━━ 5/5 1.6it/s 3.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 21.9it/s 0.0s
                   all      0.672          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.816G     0.3393         36        224: 100% ━━━━━━━━━━━━ 5/5 3.4it/s 1.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 92.1it/s 0.0s
                   all      0.922          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.816G     0.2058         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 83.0it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.816G     0.1258         36        224: 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.0it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.816G    0.08408         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 83.5it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.816G    0.06066         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 104.2it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.816G    0.04159         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.7it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.816G    0.02015         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 104.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.816G    0.04455         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.816G    0.02949         36        224: 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.6it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.816G    0.02818         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 80.3it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.816G    0.04946         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.2it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.816G   0.006518         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 103.8it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.816G    0.02011         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      15/50     0.816G    0.02175         36        224: 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 81.1it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      16/50     0.816G    0.01952         36        224: 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 104.6it/s 0.0s
                   all          1          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 6, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

16 epochs completed in 0.012 hours.
Optimizer stripped from /content/runs/classify/train/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/train/weights/best.pt, 3.0MB

Validating /content/runs/classify/train/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 28.2it/s 0.0s
                   all          1          1
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/train

✅ Training complete!
📁 Best checkpoint: /content/runs/classify/train/weights/best.pt


In [ ]:

📊 Evaluating on test split...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2866.1±1197.0 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 1.8Kit/s 0.0s
test: New cache created: /content/aerial/test.cache
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 6.2it/s 0.6s
                   all      0.984          1
Speed: 0.2ms preprocess, 2.0ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val

🎯 Test Top-1 Accuracy: 98.44%
🏆 Excellent accuracy!


In [ ]:

============================================================
🧪 exp1_imgsz320: Higher resolution (320 vs 224)
============================================================
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.1, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1_imgsz3203, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/exp1_imgsz3203, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=False, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 4225.1±1687.6 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 136.1Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1134.4±709.2 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 4.4Mit/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 320 train, 320 val
Using 2 dataloader workers
Logging results to /content/runs/classify/exp1_imgsz3203
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50     0.781G     0.5866          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 61.7it/s 0.0s
                   all      0.922          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.904G     0.2573          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.2it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.904G     0.1511          4        320: 100% ━━━━━━━━━━━━ 10/10 2.8it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.4it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.904G    0.09731          4        320: 100% ━━━━━━━━━━━━ 10/10 2.8it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.6it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.904G     0.2727          4        320: 100% ━━━━━━━━━━━━ 10/10 3.1it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.5it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.908G    0.04472          4        320: 100% ━━━━━━━━━━━━ 10/10 3.1it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 61.5it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.908G    0.02792          4        320: 100% ━━━━━━━━━━━━ 10/10 2.8it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.4it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.908G    0.09614          4        320: 100% ━━━━━━━━━━━━ 10/10 3.1it/s 3.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.2it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.908G     0.1131          4        320: 100% ━━━━━━━━━━━━ 10/10 3.1it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.6it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.908G    0.07369          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.7it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.908G    0.06447          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 46.7it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.908G    0.07815          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.7it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.908G     0.1043          4        320: 100% ━━━━━━━━━━━━ 10/10 2.9it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 63.5it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.908G    0.04522          4        320: 100% ━━━━━━━━━━━━ 10/10 3.0it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 64.4it/s 0.0s
                   all      0.984          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 4, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

14 epochs completed in 0.014 hours.
Optimizer stripped from /content/runs/classify/exp1_imgsz3203/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/exp1_imgsz3203/weights/best.pt, 3.0MB

Validating /content/runs/classify/exp1_imgsz3203/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 66.2it/s 0.0s
                   all          1          1
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/exp1_imgsz3203
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3618.9±727.7 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 17.9Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 4.8it/s 0.8s
                   all      0.984          1
Speed: 1.9ms preprocess, 1.7ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val10
  ✅ Val: 100.00% | Test: 98.44% | Epochs: 14

============================================================
🧪 exp2_sgd: SGD optimizer instead of AdamW
============================================================
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.1, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp2_sgd3, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/exp2_sgd3, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=False, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3305.5±1510.8 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 122.5Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 820.5±74.1 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 3.7Mit/s 0.0s
optimizer: SGD(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/exp2_sgd3
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50     0.725G     0.8049         36        224: 100% ━━━━━━━━━━━━ 5/5 1.6it/s 3.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 86.8it/s 0.0s
                   all      0.391          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.725G     0.7689         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 79.7it/s 0.0s
                   all      0.375          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.725G     0.7557         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.6it/s 0.0s
                   all      0.422          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.725G     0.7217         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.3it/s 0.0s
                   all      0.516          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.725G     0.6702         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.4it/s 0.0s
                   all      0.609          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.725G     0.6359         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.3it/s 0.0s
                   all       0.75          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.725G     0.5649         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 78.0it/s 0.0s
                   all      0.812          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.725G     0.5188         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.8it/s 0.0s
                   all      0.891          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.725G     0.4637         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.4it/s 0.0s
                   all      0.922          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.725G      0.415         36        224: 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.4it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.725G      0.362         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 77.1it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.725G     0.3579         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.3it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.725G     0.3006         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.8it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.725G     0.2683         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.5it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
      15/50     0.725G     0.2631         36        224: 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 104.3it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
      16/50     0.725G     0.2184         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.2it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
      17/50     0.725G     0.2103         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 90.8it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      18/50     0.727G     0.1845         36        224: 100% ━━━━━━━━━━━━ 5/5 2.0it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.7it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      19/50     0.727G     0.1709         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 79.5it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      20/50     0.727G     0.1455         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 103.6it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      21/50     0.727G     0.1523         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.4it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      22/50     0.727G     0.1279         36        224: 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 80.2it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      23/50     0.727G     0.1297         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.3it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      24/50     0.727G     0.1253         36        224: 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.4it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      25/50     0.727G     0.1213         36        224: 100% ━━━━━━━━━━━━ 5/5 1.8it/s 2.8s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 84.0it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      26/50     0.727G      0.127         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 103.8it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      27/50     0.727G     0.1038         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 83.9it/s 0.0s
                   all      0.984          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 17, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

27 epochs completed in 0.020 hours.
Optimizer stripped from /content/runs/classify/exp2_sgd3/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/exp2_sgd3/weights/best.pt, 3.0MB

Validating /content/runs/classify/exp2_sgd3/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 83.0it/s 0.0s
                   all      0.984          1
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/exp2_sgd3
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3216.7±891.1 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 15.8Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 5.6it/s 0.7s
                   all      0.984          1
Speed: 0.5ms preprocess, 1.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val11
  ✅ Val: 98.44% | Test: 98.44% | Epochs: 27

============================================================
🧪 exp3_no_augment: No mixup/cutmix/erasing (ablation)
============================================================
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.0, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp3_no_augment3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/exp3_no_augment3, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=False, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3586.7±1278.0 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 122.5Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1056.6±533.3 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 4.5Mit/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/exp3_no_augment3
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50     0.729G     0.6525         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 97.4it/s 0.0s
                   all      0.703          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.729G     0.3595         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.0it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.729G     0.1995         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.9it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.729G     0.1284         36        224: 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.0it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.729G    0.09465         36        224: 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.729G    0.05865         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 102.3it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.729G    0.04496         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.729G    0.03272         36        224: 100% ━━━━━━━━━━━━ 5/5 2.2it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.8it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.729G    0.07081         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 96.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.729G    0.01306         36        224: 100% ━━━━━━━━━━━━ 5/5 2.4it/s 2.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 78.3it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.729G    0.02266         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 3.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 78.4it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.729G     0.0544         36        224: 100% ━━━━━━━━━━━━ 5/5 2.3it/s 2.2s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.729G    0.01047         36        224: 100% ━━━━━━━━━━━━ 5/5 1.7it/s 2.9s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 96.5it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.729G     0.0244         36        224: 100% ━━━━━━━━━━━━ 5/5 2.1it/s 2.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 80.7it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      15/50     0.729G    0.04713         36        224: 100% ━━━━━━━━━━━━ 5/5 1.9it/s 2.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.0it/s 0.0s
                   all          1          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 5, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

15 epochs completed in 0.011 hours.
Optimizer stripped from /content/runs/classify/exp3_no_augment3/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/exp3_no_augment3/weights/best.pt, 3.0MB

Validating /content/runs/classify/exp3_no_augment3/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 88.2it/s 0.0s
                   all          1          1
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/exp3_no_augment3
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 2880.7±621.0 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 26.8Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 5.6it/s 0.7s
                   all      0.984          1
Speed: 0.8ms preprocess, 1.4ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val12
  ✅ Val: 100.00% | Test: 98.44% | Epochs: 15

============================================================
🧪 exp4_heavy_augment: Heavy augmentation (erasing=0.7, mixup=0.3)
============================================================
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=augmix, batch=64, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.7, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.3, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp4_heavy_augment3, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/exp4_heavy_augment3, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=False, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3048.6±999.1 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 111.3Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1502.8±1489.9 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 4.5Mit/s 0.0s
optimizer: AdamW(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 224 train, 224 val
Using 2 dataloader workers
Logging results to /content/runs/classify/exp4_heavy_augment3
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50     0.729G     0.6419         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 90.7it/s 0.0s
                   all      0.688          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.729G      0.344         36        224: 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 97.6it/s 0.0s
                   all      0.797          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.729G     0.1883         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 91.7it/s 0.0s
                   all      0.938          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.729G     0.1299         36        224: 100% ━━━━━━━━━━━━ 5/5 1.5it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.2it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.729G    0.07067         36        224: 100% ━━━━━━━━━━━━ 5/5 1.2it/s 4.1s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.5it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.729G    0.05158         36        224: 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 100.0it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.729G     0.0224         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.729G    0.04442         36        224: 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.6it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.729G    0.03011         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.729G    0.02043         36        224: 100% ━━━━━━━━━━━━ 5/5 1.5it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 101.5it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.729G    0.01544         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.4it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.729G    0.01647         36        224: 100% ━━━━━━━━━━━━ 5/5 1.5it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 98.8it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.729G   0.006364         36        224: 100% ━━━━━━━━━━━━ 5/5 1.1it/s 4.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 77.6it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.729G    0.03217         36        224: 100% ━━━━━━━━━━━━ 5/5 1.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 99.6it/s 0.0s
                   all          1          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 4, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

14 epochs completed in 0.016 hours.
Optimizer stripped from /content/runs/classify/exp4_heavy_augment3/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/exp4_heavy_augment3/weights/best.pt, 3.0MB

Validating /content/runs/classify/exp4_heavy_augment3/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 1/1 83.6it/s 0.0s
                   all          1          1
Speed: 0.1ms preprocess, 0.1ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/exp4_heavy_augment3
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3164.0±548.8 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 19.2Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 5.8it/s 0.7s
                   all      0.969          1
Speed: 0.6ms preprocess, 2.8ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to /content/runs/classify/val13
  ✅ Val: 100.00% | Test: 96.88% | Epochs: 14

============================================================
🧪 exp5_imgsz384_sgd: Largest resolution + SGD
============================================================
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.1, data=/content/aerial, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=384, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=yolov8n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp5_imgsz384_sgd2, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=10, perspective=0.0, plots=True, pose=12.0, pretrained=True, profile=False, project=None, rect=False, resume=False, retina_masks=False, rle=1.0, save=True, save_conf=False, save_crop=False, save_dir=/content/runs/classify/exp5_imgsz384_sgd2, save_frames=False, save_json=False, save_period=-1, save_txt=False, scale=0.5, seed=0, shear=0.0, show=False, show_boxes=True, show_conf=True, show_labels=True, simplify=True, single_cls=False, source=None, split=val, stream_buffer=False, task=classify, time=None, tracker=botsort.yaml, translate=0.1, val=True, verbose=False, vid_stride=1, visualize=False, warmup_bias_lr=0.1, warmup_epochs=3.0, warmup_momentum=0.8, weight_decay=0.0005, workers=2, workspace=None
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
Overriding model.yaml nc=1000 with nc=2

                   from  n    params  module                                       arguments                     
  0                  -1  1       464  ultralytics.nn.modules.conv.Conv             [3, 16, 3, 2]                 
  1                  -1  1      4672  ultralytics.nn.modules.conv.Conv             [16, 32, 3, 2]                
  2                  -1  1      7360  ultralytics.nn.modules.block.C2f             [32, 32, 1, True]             
  3                  -1  1     18560  ultralytics.nn.modules.conv.Conv             [32, 64, 3, 2]                
  4                  -1  2     49664  ultralytics.nn.modules.block.C2f             [64, 64, 2, True]             
  5                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               
  6                  -1  2    197632  ultralytics.nn.modules.block.C2f             [128, 128, 2, True]           
  7                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  8                  -1  1    460288  ultralytics.nn.modules.block.C2f             [256, 256, 1, True]           
  9                  -1  1    332802  ultralytics.nn.modules.head.Classify         [256, 2]                      
YOLOv8n-cls summary: 56 layers, 1,440,850 parameters, 1,440,850 gradients, 3.4 GFLOPs
Transferred 156/158 items from pretrained weights
AMP: running Automatic Mixed Precision (AMP) checks...
AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3733.9±1535.3 MB/s, size: 393.8 KB)
train: Scanning /content/aerial/train... 292 images, 0 corrupt: 100% ━━━━━━━━━━━━ 292/292 136.1Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1219.4±855.8 MB/s, size: 312.4 KB)
val: Scanning /content/aerial/val... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 4.1Mit/s 0.0s
optimizer: SGD(lr=0.001, momentum=0.937) with parameter groups 26 weight(decay=0.0), 27 weight(decay=0.0005), 27 bias(decay=0.0)
Image sizes 384 train, 384 val
Using 2 dataloader workers
Logging results to /content/runs/classify/exp5_imgsz384_sgd2
Starting training for 50 epochs...

      Epoch    GPU_mem       loss  Instances       Size
       1/50     0.555G     0.7783          4        384: 100% ━━━━━━━━━━━━ 19/19 5.0it/s 3.8s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 79.4it/s 0.0s
                   all      0.422          1

      Epoch    GPU_mem       loss  Instances       Size
       2/50     0.643G     0.6944          4        384: 100% ━━━━━━━━━━━━ 19/19 5.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 80.3it/s 0.0s
                   all      0.688          1

      Epoch    GPU_mem       loss  Instances       Size
       3/50     0.643G     0.5275          4        384: 100% ━━━━━━━━━━━━ 19/19 5.3it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 85.4it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
       4/50     0.643G     0.3977          4        384: 100% ━━━━━━━━━━━━ 19/19 5.8it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 87.0it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
       5/50     0.643G     0.2874          4        384: 100% ━━━━━━━━━━━━ 19/19 5.7it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 87.5it/s 0.0s
                   all      0.953          1

      Epoch    GPU_mem       loss  Instances       Size
       6/50     0.643G     0.1928          4        384: 100% ━━━━━━━━━━━━ 19/19 5.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 87.6it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       7/50     0.643G      0.148          4        384: 100% ━━━━━━━━━━━━ 19/19 5.2it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 76.5it/s 0.0s
                   all      0.969          1

      Epoch    GPU_mem       loss  Instances       Size
       8/50     0.643G     0.1148          4        384: 100% ━━━━━━━━━━━━ 19/19 5.5it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 85.8it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
       9/50     0.643G     0.1441          4        384: 100% ━━━━━━━━━━━━ 19/19 5.7it/s 3.3s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 85.7it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      10/50     0.643G    0.09433          4        384: 100% ━━━━━━━━━━━━ 19/19 5.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 74.3it/s 0.0s
                   all      0.984          1

      Epoch    GPU_mem       loss  Instances       Size
      11/50     0.643G    0.07727          4        384: 100% ━━━━━━━━━━━━ 19/19 5.6it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 74.4it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      12/50     0.643G     0.1138          4        384: 100% ━━━━━━━━━━━━ 19/19 5.3it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 78.7it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      13/50     0.643G     0.1106          4        384: 100% ━━━━━━━━━━━━ 19/19 5.5it/s 3.4s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 85.3it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      14/50     0.643G    0.09251          4        384: 100% ━━━━━━━━━━━━ 19/19 5.2it/s 3.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 69.6it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      15/50     0.643G      0.145          4        384: 100% ━━━━━━━━━━━━ 19/19 5.2it/s 3.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 86.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      16/50     0.643G    0.04392          4        384: 100% ━━━━━━━━━━━━ 19/19 5.1it/s 3.7s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 86.9it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      17/50     0.643G     0.1008          4        384: 100% ━━━━━━━━━━━━ 19/19 5.3it/s 3.6s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 73.4it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      18/50     0.643G    0.09384          4        384: 100% ━━━━━━━━━━━━ 19/19 5.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 86.1it/s 0.0s
                   all          1          1

      Epoch    GPU_mem       loss  Instances       Size
      19/50     0.643G    0.08055          4        384: 100% ━━━━━━━━━━━━ 19/19 5.4it/s 3.5s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 87.7it/s 0.0s
                   all          1          1
EarlyStopping: Training stopped early as no improvement observed in last 10 epochs. Best results observed at epoch 9, best model saved as best.pt.
To update EarlyStopping(patience=10) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

19 epochs completed in 0.019 hours.
Optimizer stripped from /content/runs/classify/exp5_imgsz384_sgd2/weights/last.pt, 3.0MB
Optimizer stripped from /content/runs/classify/exp5_imgsz384_sgd2/weights/best.pt, 3.0MB

Validating /content/runs/classify/exp5_imgsz384_sgd2/weights/best.pt...
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 2/2 50.0it/s 0.0s
                   all          1          1
Speed: 0.2ms preprocess, 0.4ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/exp5_imgsz384_sgd2
Ultralytics 8.4.22 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (NVIDIA L4, 22563MiB)
YOLOv8n-cls summary (fused): 30 layers, 1,437,442 parameters, 0 gradients, 3.3 GFLOPs
train: /content/aerial/train... found 292 images in 2 classes ✅ 
val: /content/aerial/val... found 64 images in 2 classes ✅ 
test: /content/aerial/test... found 64 images in 2 classes ✅ 
test: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3360.4±1435.6 MB/s, size: 388.2 KB)
test: Scanning /content/aerial/test... 64 images, 0 corrupt: 100% ━━━━━━━━━━━━ 64/64 19.2Mit/s 0.0s
               classes   top1_acc   top5_acc: 100% ━━━━━━━━━━━━ 4/4 3.8it/s 1.1s
                   all      0.984          1
Speed: 1.0ms preprocess, 4.6ms inference, 0.0ms loss, 0.0ms postprocess per image
Results saved to /content/runs/classify/val14
  ✅ Val: 100.00% | Test: 98.44% | Epochs: 19

===========================================================================
📊 EXPERIMENT SUMMARY
===========================================================================
Experiment             Description                           Val%  Test%   Ep
---------------------------------------------------------------------------
baseline               imgsz=224, AdamW, cos_lr, randaugme 100.0%  98.4%   20
exp1_imgsz320          Higher resolution (320 vs 224)      100.0%  98.4%   14
exp2_sgd               SGD optimizer instead of AdamW       98.4%  98.4%   27
exp3_no_augment        No mixup/cutmix/erasing (ablation)  100.0%  98.4%   15
exp4_heavy_augment     Heavy augmentation (erasing=0.7, mi 100.0%  96.9%   14
exp5_imgsz384_sgd      Largest resolution + SGD            100.0%  98.4%   19
===========================================================================

🏆 Best experiment: baseline (Test: 98.44%)


In [ ]:
🔍 Sample predictions on test images:
  ✅ True: flat     | Pred: flat     | Conf: 99.95% | pic122.png
  ✅ True: flat     | Pred: flat     | Conf: 99.86% | pic66.png
  ✅ True: flat     | Pred: flat     | Conf: 99.98% | pic134.png
  ✅ True: pitched  | Pred: pitched  | Conf: 100.00% | pic6.png
  ✅ True: pitched  | Pred: pitched  | Conf: 99.99% | pic149.png
  ✅ True: pitched  | Pred: pitched  | Conf: 100.00% | pic98.png
